In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("case-study").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 03:44:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 03:44:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [6]:
import pandas as pd
df1 = pd.read_csv("data/customers.csv")
df1.head()

,customer_id,customer_name,city,state,registration_date,customer_segment
0,1,Customer_1,Columbus,OH,2023-10-17,VIP
1,2,Customer_2,Miami,CA,2022-04-25,Premium
2,3,Customer_3,Atlanta,FL,2022-01-26,Premium
3,4,Customer_4,Chicago,OH,2022-10-09,Standard
4,5,Customer_5,Columbus,IL,2022-09-08,Premium


In [50]:
df1 = pd.read_csv("data/orders.csv")
df1.head()

,order_id,customer_id,order_date,payment_mode,order_status
0,1,54630,2024-01-25,Credit Card,Delivered
1,2,22415,2024-07-08,Credit Card,Delivered
2,3,20909,2024-01-23,UPI,Delivered
3,4,98027,2024-03-02,Credit Card,Cancelled
4,5,31332,2024-12-17,UPI,Cancelled


In [33]:

customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
count = customers.count()
print("Number of returns:", count)

Number of returns: 100000


In [41]:

df = spark.read.csv("data/products.csv", header=True, inferSchema=True)
count = products.count()
print("Number of returns:", count)
products.take(10)

Number of returns: 50000


[Row(product_id=1, product_name='Product_1', category='Home & Kitchen', brand='Brand_A', unit_cost=509.39),
 Row(product_id=2, product_name='Product_2', category='Electronics', brand='Brand_A', unit_cost=332.22),
 Row(product_id=3, product_name='Product_3', category='Sports', brand='Brand_D', unit_cost=68.58),
 Row(product_id=4, product_name='Product_4', category='Clothing', brand='Brand_A', unit_cost=729.19),
 Row(product_id=5, product_name='Product_5', category='Home & Kitchen', brand='Brand_D', unit_cost=326.77),
 Row(product_id=6, product_name='Product_6', category='Beauty', brand='Brand_E', unit_cost=684.21),
 Row(product_id=7, product_name='Product_7', category='Toys', brand='Brand_D', unit_cost=634.58),
 Row(product_id=8, product_name='Product_8', category='Home & Kitchen', brand='Brand_B', unit_cost=158.47),
 Row(product_id=9, product_name='Product_9', category='Electronics', brand='Brand_C', unit_cost=287.9),
 Row(product_id=10, product_name='Product_10', category='Clothing', 

In [ ]:

df = spark.read.csv("data/returns.csv", header=True, inferSchema=True)
count = products.count()
print("Number of returns:", count)
products.take(10)

In [3]:

returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)
count = returns.count()
print("Number of returns:", count)
returns.take(10)

Number of returns: 100000


[Row(return_id=1, order_id=391349, return_date=datetime.date(2024, 8, 6), return_reason='Changed Mind'),
 Row(return_id=2, order_id=456171, return_date=datetime.date(2024, 6, 28), return_reason='Arrived Damaged'),
 Row(return_id=3, order_id=977675, return_date=datetime.date(2024, 9, 26), return_reason='Size Issue'),
 Row(return_id=4, order_id=80452, return_date=datetime.date(2024, 7, 30), return_reason='Arrived Damaged'),
 Row(return_id=5, order_id=10920, return_date=datetime.date(2024, 6, 15), return_reason='Size Issue'),
 Row(return_id=6, order_id=365396, return_date=datetime.date(2024, 8, 16), return_reason='Wrong Item Shipped'),
 Row(return_id=7, order_id=675259, return_date=datetime.date(2024, 10, 2), return_reason='Arrived Damaged'),
 Row(return_id=8, order_id=871061, return_date=datetime.date(2024, 11, 2), return_reason='Defective Product'),
 Row(return_id=9, order_id=996307, return_date=datetime.date(2024, 4, 11), return_reason='Defective Product'),
 Row(return_id=10, order_id=


orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
count = orders.count()
print("Number of returns:", count)

Q2 find the total sales amount generated by each product catagory

In [57]:
import os
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
import pandas as pd

# 1. Initialize SparkSession (Missing in original script)
spark = SparkSession.builder \
    .appName("CategorySalesAnalysis") \
    .getOrCreate()

# 2. Load the CSV datasets into DataFrames
order_items_df = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
products_df = spark.read.csv("data/products.csv", header=True, inferSchema=True)

# 3. Calculate the line item total sales (quantity * selling_price)
order_items_df = order_items_df.withColumn(
    "sales_amount", 
    F.col("quantity") * F.col("selling_price")
)

# 4. Join the datasets on 'product_id'
joined_df = order_items_df.join(products_df, on="product_id", how="inner")

# 5. Group by 'category', sum the sales, and round to 2 decimal places
category_sales_df = joined_df.groupBy("category") \
    .agg(F.round(F.sum("sales_amount"), 2).alias("total_sales_amount")) \
    .orderBy(F.col("total_sales_amount").desc())

# 6. Display the results
category_sales_df.show(truncate=False)

# 7. Convert PySpark DataFrame to Pandas DataFrame correctly
category_sales = category_sales_df.toPandas()

# Ensure output directory exists
output_dir = "output"


# Save to CSV
output_path = os.path.join(output_dir, "category_sales.csv")
category_sales.to_csv(output_path, index=False)

print(f"Data exported successfully to {output_path}")

26/06/15 06:08:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+--------------+------------------+
|category      |total_sales_amount|
+--------------+------------------+
|Beauty        |7.626693059E8     |
|Home & Kitchen|7.5813887328E8    |
|Books         |7.4649077835E8    |
|Toys          |7.446190723E8     |
|Electronics   |7.4426650411E8    |
|Sports        |7.4333886813E8    |
|Clothing      |7.4192279457E8    |
+--------------+------------------+



Data exported successfully to output/category_sales.csv


In [61]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os

# 1. Initialize a Spark session
spark = SparkSession.builder \
    .appName("Top10Customers") \
    .getOrCreate()


# Define file paths
order_items_path = os.path.join(DATA_DIR, "data/order_items.csv")
orders_path = os.path.join(DATA_DIR, "data/orders.csv")  # Requires the mapping file
customers_path = os.path.join(DATA_DIR, "data/customers.csv")

try:
    # 3. Load the datasets into PySpark DataFrames
    order_items_df = spark.read.csv(order_items_path, header=True, inferSchema=True)
    orders_df = spark.read.csv(orders_path, header=True, inferSchema=True)
    customers_df = spark.read.csv(customers_path, header=True, inferSchema=True)

    # 4. Calculate total purchase amount for each line item (quantity * selling_price)
    order_items_df = order_items_df.withColumn(
        "line_total", 
        F.col("quantity") * F.col("selling_price")
    )

    # 5. Join order_items with orders to get the customer_id associated with each order
    orders_with_customers = order_items_df.join(orders_df, on="order_id", how="inner")

    # 6. Group by customer_id and sum the line totals
    customer_spending = orders_with_customers.groupBy("customer_id") \
        .agg(F.round(F.sum("line_total"), 2).alias("total_purchase_amount"))

    # 7. Join with customers table to fetch customer names
    top_customers_df = customer_spending.join(customers_df, on="customer_id", how="inner")

    # 8. Order by total purchase amount in descending order and select top 10
    result_df = top_customers_df.select("customer_id", "customer_name", "total_purchase_amount") \
        .orderBy(F.col("total_purchase_amount").desc()) \
        .limit(10)

    # 9. Show the final results
    result_df.show(truncate=False)
    result_df.write.mode("overwrite").parquet("output/parquet_data")

except Exception as e:
    print(f"Error occurred: {e}")
    print("\nPlease verify that 'order_items.csv', 'orders.csv', and 'customers.csv' all exist in the specified directory.")

+-----------+--------------+---------------------+
|customer_id|customer_name |total_purchase_amount|
+-----------+--------------+---------------------+
|93094      |Customer_93094|181569.68            |
|64560      |Customer_64560|169060.4             |
|23289      |Customer_23289|161573.8             |
|52275      |Customer_52275|153364.79            |
|61218      |Customer_61218|153067.55            |
|52034      |Customer_52034|152680.05            |
|40442      |Customer_40442|151037.32            |
|60528      |Customer_60528|148691.95            |
|84830      |Customer_84830|148363.84            |
|82593      |Customer_82593|148281.04            |
+-----------+--------------+---------------------+



In [63]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("MonthlySalesTrends") \
    .getOrCreate()

# 2. Read the datasets
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True) # Contains order_id, order_date

# 3. Calculate line sales amount (quantity * selling_price)
order_items = order_items.withColumn("sales_amount", F.col("quantity") * F.col("selling_price"))

# 4. Join datasets together
# Join items to orders to get 'order_date', and to products to get 'category'
joined_df = order_items.join(orders, on="order_id", how="inner") \
                       .join(products, on="product_id", how="inner")

# 5. Extract Month-Year format (YYYY-MM) from the order_date
# (Handles both DateType/TimestampType or string format)
trends_df = joined_df.withColumn("month", F.date_format(F.col("order_date"), "yyyy-MM"))

# 6. Aggregate sales amount by category and month
monthly_trends = trends_df.groupBy("category", "month") \
    .agg(F.round(F.sum("sales_amount"), 2).alias("total_sales")) \
    .orderBy("category", "month")

# 7. Display the result
monthly_trends.show(truncate=False)

# 8. Save the data to an output folder as a single CSV file
monthly_trends.coalesce(1).write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("output/monthly_sales_trends")

+--------+-------+-------------+
|category|month  |total_sales  |
+--------+-------+-------------+
|Beauty  |2024-01|6.444743596E7|
|Beauty  |2024-02|6.038332564E7|
|Beauty  |2024-03|6.388434236E7|
|Beauty  |2024-04|6.243401371E7|
|Beauty  |2024-05|6.454298305E7|
|Beauty  |2024-06|6.358210818E7|
|Beauty  |2024-07|6.435022315E7|
|Beauty  |2024-08|6.404933883E7|
|Beauty  |2024-09|6.206964965E7|
|Beauty  |2024-10|6.406140866E7|
|Beauty  |2024-11|6.370936426E7|
|Beauty  |2024-12|6.515511245E7|
|Books   |2024-01|6.320723893E7|
|Books   |2024-02|5.867111637E7|
|Books   |2024-03|6.316039832E7|
|Books   |2024-04|6.086804513E7|
|Books   |2024-05|6.356037914E7|
|Books   |2024-06|6.090605764E7|
|Books   |2024-07|6.320803776E7|
|Books   |2024-08|6.257991017E7|
+--------+-------+-------------+
only showing top 20 rows



Q5 : find the return percentage for each products catagorys


In [3]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("MonthlySalesTrends") \
    .getOrCreate()

order_items_df = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
products_df= spark.read.csv("data/products.csv", header=True, inferSchema=True)
returns_df = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

# Ensure your views are registered
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")
returns_df.createOrReplaceTempView("returns")

return_pct_sql = spark.sql("""
    SELECT 
        p.category,
        COUNT(oi.order_item_id) AS total_ordered_items,
        COUNT(r.return_id) AS total_returned_items,
        ROUND((COUNT(r.return_id) / COUNT(oi.order_item_id)) * 100, 2) AS return_percentage
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN returns r ON oi.order_id = r.order_id
    GROUP BY p.category
    ORDER BY return_percentage DESC
""")

return_pct_sql.show()

26/06/16 05:07:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
                                                                                

+--------------+-------------------+--------------------+-----------------+
|      category|total_ordered_items|total_returned_items|return_percentage|
+--------------+-------------------+--------------------+-----------------+
|          Toys|             430418|               43382|            10.08|
|        Beauty|             430547|               43194|            10.03|
|        Sports|             424412|               42530|            10.02|
|         Books|             427086|               42809|            10.02|
|Home & Kitchen|             434034|               43418|             10.0|
|   Electronics|             425896|               42601|             10.0|
|      Clothing|             427607|               42660|             9.98|
+--------------+-------------------+--------------------+-----------------+



Q6 ; deterinme the most perferred payment mode in each state


In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("most prefered payment methord") \
    .getOrCreate()

customers_df = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
orders_df = spark.read.csv("data/orders.csv", header=True, inferSchema=True)



payment_df= customers_df.join(orders_df, on="customer_id", how="inner")
payment_df.show()

payment_df.createOrReplaceTempView("payment")

## Q6) Determine the most preferred payment mode in each state.

preferred_payment_sql = spark.sql("""
    WITH ranked_payments AS (
        SELECT 
            state, 
            payment_mode, 
            COUNT(*) AS total_orders,
            ROW_NUMBER() OVER (PARTITION BY state ORDER BY COUNT(*) DESC) as rank
        FROM payment
        GROUP BY state, payment_mode
    )
   
    SELECT state, payment_mode, total_orders
    FROM ranked_payments
    WHERE rank = 1
    ORDER BY state
""")

preferred_payment_sql.show()

+-----------+--------------+-----------+-----+-----------------+----------------+--------+----------+----------------+------------+
|customer_id| customer_name|       city|state|registration_date|customer_segment|order_id|order_date|    payment_mode|order_status|
+-----------+--------------+-----------+-----+-----------------+----------------+--------+----------+----------------+------------+
|      54630|Customer_54630|    Houston|   NC|       2023-01-04|             VIP|       1|2024-01-25|     Credit Card|   Delivered|
|      22415|Customer_22415|    Detroit|   MI|       2022-10-24|        Standard|       2|2024-07-08|     Credit Card|   Delivered|
|      20909|Customer_20909|    Chicago|   GA|       2023-10-14|        Standard|       3|2024-01-23|             UPI|   Delivered|
|      98027|Customer_98027|  Charlotte|   IL|       2022-09-30|         Premium|       4|2024-03-02|     Credit Card|   Cancelled|
|      31332|Customer_31332|Los Angeles|   FL|       2023-09-16|         Pre

[Stage 15:=============================>                            (1 + 1) / 2]

+-----+----------------+------------+
|state|    payment_mode|total_orders|
+-----+----------------+------------+
|   CA|             UPI|       20246|
|   FL|      Debit Card|       20010|
|   GA|     Net Banking|       20041|
|   IL|Cash on Delivery|       20498|
|   MI|      Debit Card|       20416|
|   NC|     Net Banking|       19596|
|   NY|      Debit Card|       20369|
|   OH|     Net Banking|       20351|
|   TX|             UPI|       20065|
|   WA|             UPI|       20244|
+-----+----------------+------------+



Q7 : identify customers wjo purchased products  form at lets 5 different categories  and spent more than 100,000